In [2]:
import pyspark.sql as sql
import pyspark.sql.functions as F
import pyspark.sql.types as T
import os

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

In [ ]:
spark : sql.SparkSession = (sql.SparkSession.builder
	.appName("PlatformsETL")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

# Data extraction

In [21]:
df : sql.DataFrame = spark.read.csv("data/stations_platforms.csv", header=True, sep=";", inferSchema=True)
df.show(5, truncate=False)

+-------+---------------+--------------------+--------------+----------+------------+-------------+---------------------+------------------------+-----------------------+--------+---------------+---------------------+----------------------+-------------------+------------------------+------------------------------------+
|ID quai|Stopplaats naam|Nom du point d'arrêt|Numéro du quai|Perrontype|Type de quai|Platform type|Nominale perronhoogte|Hauteur nominale du quai|Platform nominal height|PTCAR ID|Type stopplaats|Type de point d'arrêt|Type of stopping point|Area               |Arrondissement          |Geo Point                           |
+-------+---------------+--------------------+--------------+----------+------------+-------------+---------------------+------------------------+-----------------------+--------+---------------+---------------------+----------------------+-------------------+------------------------+------------------------------------+
|32020  |ANDERLECHT     |ANDERL

In [5]:
df.printSchema()

root
 |-- ID quai: integer (nullable = true)
 |-- Stopplaats naam: string (nullable = true)
 |-- Nom du point d'arrêt: string (nullable = true)
 |-- Numéro du quai: integer (nullable = true)
 |-- Perrontype: string (nullable = true)
 |-- Type de quai: string (nullable = true)
 |-- Platform type: string (nullable = true)
 |-- Nominale perronhoogte: string (nullable = true)
 |-- Hauteur nominale du quai: string (nullable = true)
 |-- Platform nominal height: string (nullable = true)
 |-- PTCAR ID: integer (nullable = true)
 |-- Type stopplaats: string (nullable = true)
 |-- Type de point d'arrêt: string (nullable = true)
 |-- Type of stopping point: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- Arrondissement: string (nullable = true)
 |-- Geo Point: string (nullable = true)



# Data validation

## Null values

In [6]:
null_counts = []
for column in df.columns:
    null_count = df.filter(F.col(column).isNull()).count()
    null_counts.append((column, null_count))
for col_name, null_count in null_counts:
    print(f"Column '{col_name}' has {null_count} null values")

Column 'ID quai' has 0 null values
Column 'Stopplaats naam' has 2 null values
Column 'Nom du point d'arrêt' has 2 null values
Column 'Numéro du quai' has 14 null values
Column 'Perrontype' has 0 null values
Column 'Type de quai' has 0 null values
Column 'Platform type' has 0 null values
Column 'Nominale perronhoogte' has 0 null values
Column 'Hauteur nominale du quai' has 0 null values
Column 'Platform nominal height' has 0 null values
Column 'PTCAR ID' has 2 null values
Column 'Type stopplaats' has 0 null values
Column 'Type de point d'arrêt' has 0 null values
Column 'Type of stopping point' has 0 null values
Column 'Area' has 0 null values
Column 'Arrondissement' has 0 null values
Column 'Geo Point' has 2 null values


## Unique values

In [7]:
unique_counts = []
for column in df.columns :
    unique_count : int = df.agg(F.count_distinct(column).alias('count')).collect()[0]['count']
    unique_counts.append((column, unique_count))
for col_name, unique_count in unique_counts :
    print(f"Column {col_name} has {unique_count} unique values.")

Column ID quai has 1560 unique values.
Column Stopplaats naam has 553 unique values.
Column Nom du point d'arrêt has 553 unique values.
Column Numéro du quai has 24 unique values.
Column Perrontype has 2 unique values.
Column Type de quai has 2 unique values.
Column Platform type has 2 unique values.
Column Nominale perronhoogte has 3 unique values.
Column Hauteur nominale du quai has 3 unique values.
Column Platform nominal height has 3 unique values.
Column PTCAR ID has 553 unique values.
Column Type stopplaats has 2 unique values.
Column Type de point d'arrêt has 2 unique values.
Column Type of stopping point has 2 unique values.
Column Area has 5 unique values.
Column Arrondissement has 20 unique values.
Column Geo Point has 553 unique values.


## Empty values

In [8]:
empty_counts = []
string_columns = [f.name for f in df.schema.fields if isinstance(f.dataType, T.StringType)]
for column in string_columns:
    empty_count = df.filter(F.col(column) == "").count()
    empty_counts.append((column, empty_count))
for col_name, empty_count in empty_counts:
    print(f"Column '{col_name}' has {empty_count} empty values")

Column 'Stopplaats naam' has 0 empty values
Column 'Nom du point d'arrêt' has 0 empty values
Column 'Perrontype' has 0 empty values
Column 'Type de quai' has 0 empty values
Column 'Platform type' has 0 empty values
Column 'Nominale perronhoogte' has 0 empty values
Column 'Hauteur nominale du quai' has 0 empty values
Column 'Platform nominal height' has 0 empty values
Column 'Type stopplaats' has 0 empty values
Column 'Type de point d'arrêt' has 0 empty values
Column 'Type of stopping point' has 0 empty values
Column 'Area' has 0 empty values
Column 'Arrondissement' has 0 empty values
Column 'Geo Point' has 0 empty values


## Duplicate values

In [9]:
duplicates = df.groupBy(df.columns).count().filter(F.col("count") > 1)
duplicate_count = duplicates.count()
if duplicate_count > 0:
    print(f"There are {duplicate_count} duplicate lines in the data")
else:
    print("No duplicate lines found in the data")

No duplicate lines found in the data


# Data transformation

## Removing uncessecary columns

In [22]:
new_df = df.drop(
	"Stopplaats naam",
	"Perrontype",
	"Type de quai",
	"Nominale perronhoogte",
	"Hauteur nominale du quai",
	"Platform nominal height",
	"Type stopplaats",
	"Type de point d'arrêt",
	"Type of stopping point",
	"Area",
	"Arrondissement"
)
new_df.show(5, truncate = False)

+-------+--------------------+--------------+-------------+--------+------------------------------------+
|ID quai|Nom du point d'arrêt|Numéro du quai|Platform type|PTCAR ID|Geo Point                           |
+-------+--------------------+--------------+-------------+--------+------------------------------------+
|32020  |ANDERLECHT          |1             |Lateral      |2089    |50.81791888922866, 4.291539354096092|
|32021  |ANDERLECHT          |4             |Lateral      |2089    |50.81791888922866, 4.291539354096092|
|32014  |OUGREE              |2             |Lateral      |2011    |50.60537053844519, 5.536956573101103|
|32013  |OUGREE              |1             |Lateral      |2011    |50.60537053844519, 5.536956573101103|
|32001  |TOUR ET TAXIS       |2             |Lateral      |1979    |50.87155655860092, 4.340863745556818|
+-------+--------------------+--------------+-------------+--------+------------------------------------+
only showing top 5 rows


## Renaming the columns

In [23]:
new_df = (
    new_df.withColumnRenamed("ID Quai", "ID")
	.withColumnRenamed("Nom du point d'arrêt", "Station_name")
	.withColumnRenamed("Numéro du quai", "Platform_number")
	.withColumnRenamed("Platform type", "Platform_type")
	.withColumnRenamed("PTCAR ID", "Station_ID")
)
new_df.show(5, truncate = False)

+-----+-------------+---------------+-------------+----------+------------------------------------+
|ID   |Station_name |Platform_number|Platform_type|Station_ID|Geo Point                           |
+-----+-------------+---------------+-------------+----------+------------------------------------+
|32020|ANDERLECHT   |1              |Lateral      |2089      |50.81791888922866, 4.291539354096092|
|32021|ANDERLECHT   |4              |Lateral      |2089      |50.81791888922866, 4.291539354096092|
|32014|OUGREE       |2              |Lateral      |2011      |50.60537053844519, 5.536956573101103|
|32013|OUGREE       |1              |Lateral      |2011      |50.60537053844519, 5.536956573101103|
|32001|TOUR ET TAXIS|2              |Lateral      |1979      |50.87155655860092, 4.340863745556818|
+-----+-------------+---------------+-------------+----------+------------------------------------+
only showing top 5 rows


## Converting and renaming the column "Geo Point"

In [18]:
new_df.filter(F.col("Geo Point").isNull()).show(5, truncate=False)

+-----------+------------+---------------+-------------+----------+---------+
|Platform_ID|Station_name|Platform_number|Platform_type|Station_ID|Geo Point|
+-----------+------------+---------------+-------------+----------+---------+
|30326      |NULL        |1              |Lateral      |NULL      |NULL     |
|30327      |NULL        |2              |Lateral      |NULL      |NULL     |
+-----------+------------+---------------+-------------+----------+---------+



In [24]:
coords = F.split(F.col("Geo Point"), ",")

new_df = (
    new_df.withColumn("X", F.floor(coords.getItem(0).cast("double") * 1e6) / 1e6) 
    .withColumn("Y", F.floor(coords.getItem(1).cast("double") * 1e6) / 1e6)
    .drop("Geo Point")
)
new_df.show(5, truncate=False)

+-----+-------------+---------------+-------------+----------+---------+--------+
|ID   |Station_name |Platform_number|Platform_type|Station_ID|X        |Y       |
+-----+-------------+---------------+-------------+----------+---------+--------+
|32020|ANDERLECHT   |1              |Lateral      |2089      |50.817918|4.291539|
|32021|ANDERLECHT   |4              |Lateral      |2089      |50.817918|4.291539|
|32014|OUGREE       |2              |Lateral      |2011      |50.60537 |5.536956|
|32013|OUGREE       |1              |Lateral      |2011      |50.60537 |5.536956|
|32001|TOUR ET TAXIS|2              |Lateral      |1979      |50.871556|4.340863|
+-----+-------------+---------------+-------------+----------+---------+--------+
only showing top 5 rows


In [25]:
new_df.filter(F.col("X").isNull() | F.col("Y").isNull()).show(5, truncate=False)

+-----+------------+---------------+-------------+----------+----+----+
|ID   |Station_name|Platform_number|Platform_type|Station_ID|X   |Y   |
+-----+------------+---------------+-------------+----------+----+----+
|30326|NULL        |1              |Lateral      |NULL      |NULL|NULL|
|30327|NULL        |2              |Lateral      |NULL      |NULL|NULL|
+-----+------------+---------------+-------------+----------+----+----+



## Convert the columns to the right datatypes

In [26]:
new_df = (
    new_df.withColumn("ID", F.col("ID").cast("integer"))
    .withColumn("Station_name", F.col("Station_name").cast("String"))
    .withColumn("Platform_number", F.col("Platform_number").cast("integer"))
    .withColumn("Platform_type", F.col("Platform_type").cast("string"))
)
new_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Station_name: string (nullable = true)
 |-- Platform_number: integer (nullable = true)
 |-- Platform_type: string (nullable = true)
 |-- Station_ID: integer (nullable = true)
 |-- X: double (nullable = true)
 |-- Y: double (nullable = true)



## Reorder the columns

In [27]:
final_df = new_df.select(
    "ID",
    "Station_ID",
    "Station_name",
    "Platform_type",
    "X",
    "Y"
)
final_df.show(5, truncate=False)

+-----+----------+-------------+-------------+---------+--------+
|ID   |Station_ID|Station_name |Platform_type|X        |Y       |
+-----+----------+-------------+-------------+---------+--------+
|32020|2089      |ANDERLECHT   |Lateral      |50.817918|4.291539|
|32021|2089      |ANDERLECHT   |Lateral      |50.817918|4.291539|
|32014|2011      |OUGREE       |Lateral      |50.60537 |5.536956|
|32013|2011      |OUGREE       |Lateral      |50.60537 |5.536956|
|32001|1979      |TOUR ET TAXIS|Lateral      |50.871556|4.340863|
+-----+----------+-------------+-------------+---------+--------+
only showing top 5 rows
